In [ ]:
# Importar las librerías necesarias
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [ ]:
# Cargar las matrices de características de entrenamiento y prueba
X_train = pd.read_parquet('../data/X_train.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/y_train.parquet', engine='fastparquet')['is_fraud']
y_test = pd.read_parquet('../data/y_test.parquet', engine='fastparquet')['is_fraud']
    
# Verificar las dimensiones de los conjuntos de datos
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

In [ ]:
# Escalado de características: la Regresión Logística es sensible a las diferencias de escala
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Inicializar Regresión Logística usando 'class_weight=balanced' para tratar el desbalance de clases
logic_model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Entrenar el modelo base (baseline)
logic_model.fit(X_train_scaled, y_train)

In [ ]:
# Generar predicciones en el conjunto de prueba
y_pred = logic_model.predict(X_test_scaled)

# Mostrar la matriz de confusión y el reporte de clasificación
print("=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

**Evaluación de la Regresión Logística**: La precisión y el F1-Score para la clase positiva (fraude) resultaron insuficientes para un entorno de producción. Se cambia la estrategia a XGBoost para capturar fronteras de decisión no lineales.

In [ ]:
# Calcular el factor de peso de clases para compensar el desbalance (Casos Negativos / Casos Positivos)
class_counts = y_train.value_counts()
scale_weight = class_counts[0] / class_counts[1]

# Inicializar XGBoost con configuración para balanceo de clases
xgb_model = XGBClassifier(
    n_estimators=300, 
    max_depth=5, 
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

In [ ]:
# Validación cruzada con k-folds

# Queremos probar distintos umbrales para luego elegir el mejor en base a f1-scores
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

pr_auc_scores = []
# Para f1-scores usamos una lista ya que queremos guardar cada métrica de cada fold de cada umbral
f1_scores = {threshold: [] for threshold in thresholds}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# train_idx: índices de las filas para entrenar y val_idx índices de las filas para validar.
for train_idx, val_idx in skf.split(X_train, y_train):

    # Separar datos de entrenamiento y validación
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]

    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # Entrenar XGBoost en el fold actual
    xgb_model.fit(X_tr, y_tr)

    # Obtener probabilidades de fraude      
    probs = xgb_model.predict_proba(X_val)[:, 1]

    # PR-AUC
    pr_auc_scores.append(
        average_precision_score(y_val, probs)
    )

    # Evaluar F1 para cada threshold
    for threshold in thresholds:
        # Convertimos la probabilidad en 0 o 1 (dependerá del threshold)
        preds = (probs >= threshold).astype(int)

        f1_scores[threshold].append(
            f1_score(y_val, preds)
        )


# Mostrar resultados promedio
print("=== CROSS-VALIDATION RESULTS (5-FOLD) ===")

print(
    f"Mean PR-AUC: "
    f"{np.mean(pr_auc_scores):.4f} ± {np.std(pr_auc_scores):.4f}"
)

# Vamos a mostrar los f1-scores de cada threshold
for threshold in thresholds:
    print(
        f"F1 (Threshold {threshold:.2f}): "
        f"{np.mean(f1_scores[threshold]):.4f} ± "
        f"{np.std(f1_scores[threshold]):.4f}"
    )


# Seleccionar el threshold con mayor F1 promedio
mean_f1 = {
    threshold: np.mean(f1_scores[threshold])
    for threshold in thresholds
}

best_threshold = max(mean_f1, key=mean_f1.get)

print("\n=== BEST THRESHOLD ===")
print(f"Threshold seleccionado: {best_threshold:.2f}")

In [ ]:
# Entrenar el modelo final con todos los datos de entrenamiento
xgb_model.fit(X_train, y_train)

# Obtener probabilidades de fraude en el conjunto de test
y_probs_test = xgb_model.predict_proba(X_test)[:, 1]

# Aplicar el threshold seleccionado durante la validación
y_pred_test = (y_probs_test >= best_threshold).astype(int)

# Evaluación final sobre el conjunto de test
print("\n=== FINAL TEST RESULTS ===")
print(f"Threshold: {best_threshold:.2f}")

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred_test))

print("\nCLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_test))

In [ ]:
# Crear una copia de X_test para el dashboard
df_dashboard = X_test.copy()

# Agregar la etiqueta real
df_dashboard['is_fraud_real'] = y_test.values

# Agregar las probabilidades de fraude predichas por XGBoost
df_dashboard['fraud_probability'] = y_probs_test

# Agregar las predicciones usando el threshold seleccionado
df_dashboard['model_prediction'] = y_pred_test

# Agregar el threshold utilizado por el modelo
df_dashboard['threshold'] = best_threshold

# Agregar un indicador de error
# 1 = predicción incorrecta, 0 = predicción correcta
df_dashboard['prediction_error'] = (
    df_dashboard['is_fraud_real'] != df_dashboard['model_prediction']
).astype(int)

# Exportar el dataframe para Power BI
df_dashboard.to_csv(
    '../data/fraud_results_powerbi.csv',
    index=False
)

print("File 'fraud_results_powerbi.csv' generated successfully!")
print(f"Total records exported for dashboarding: {len(df_dashboard)}")

### Justificación del Modelo y Estrategia de Decisión

**Modelo Base con Regresión Logística:**
Se utilizó como punto de comparación (benchmark). Aunque `class_weight='balanced'` mejoró la detección de fraudes, el modelo generó demasiados falsos positivos, resultando en una precisión y F1-Score muy bajos para la clase positiva.

**Ventajas de XGBoost y Gradient Boosting:**
XGBoost permite capturar relaciones complejas y no lineales entre las variables. Además, `scale_pos_weight` permite compensar el fuerte desbalance entre transacciones legítimas y fraudulentas.

**Optimización del Umbral de Decisión:**
En lugar de utilizar el umbral estándar de 0.50, se evaluaron distintos umbrales mediante validación cruzada. El mejor resultado se obtuvo con un threshold de 0.80, mejorando el equilibrio entre precisión y recall.

**Validación Cruzada Estratificada (Stratified K-Fold):**
Debido al fuerte desbalance de clases (~0.52% de casos positivos), se utilizó StratifiedKFold con 5 particiones para mantener una proporción similar de clases en cada fold y comprobar la estabilidad del modelo mediante PR-AUC y F1-Score.

**Evaluación Final:**
Una vez seleccionado el modelo y el threshold, XGBoost se reentrenó utilizando todos los datos de entrenamiento y se evaluó sobre el conjunto de test, reservado para medir el rendimiento final sobre datos no utilizados durante el entrenamiento ni la selección del threshold.